In [15]:
import pandas as pd
import psycopg2

In [16]:
def connet_to_database():
    try: 
        conn = psycopg2.connect(
            host = "localhost",
            database =  "weather_data",
            user = "postgres",
            password = "Volume27@",
            port = "5432"
        )

        cursor = conn.cursor()

    except Exception as error:
        print(f"Database connection failed: {error}")
        return None, None

    return conn, cursor

# Connect to database
conn, cur = connet_to_database()

In [17]:
def insert_data(chunk):
    query = """
    INSERT INTO weather_data (
        Location,
        Date_Time,
        Temperature_C,
        Humidity_pct,
        Precipitation_mm,
        Wind_Speed_kmh
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    """

    cur.executemany(query, chunk)

    conn.commit()

In [18]:
df = pd.read_csv("weather_data.csv")
df.shape

(1000000, 6)

In [19]:
index_list = list()
step = df.shape[0]/20
for i in range(0,int(df.shape[0]), int(step)):
    index = (i, i + int(step))
    index_list.append(index)

In [20]:
for i in index_list:
    start = i[0]
    stop = i[1]
    chunk = df.iloc[start:stop]
    data = list(chunk.itertuples(index=False, name=None))
    insert_data(data)

In [21]:
df.head()

,Location,Date_Time,Temperature_C,Humidity_pct,Precipitation_mm,Wind_Speed_kmh
0,San Diego,2024-01-14 21:12:46,10.683001,41.195754,4.020119,8.233540
1,San Diego,2024-05-17 15:22:10,8.734140,58.319107,9.111623,27.715161
2,San Diego,2024-05-11 09:30:59,11.632436,38.820175,4.607511,28.732951
3,Philadelphia,2024-02-26 17:32:39,-8.628976,54.074474,3.183720,26.367303
4,San Antonio,2024-04-29 13:23:51,39.808213,72.899908,9.598282,29.898622


In [22]:
df.columns

Index(['Location', 'Date_Time', 'Temperature_C', 'Humidity_pct',
       'Precipitation_mm', 'Wind_Speed_kmh'],
      dtype='str')